# Modèle climatique : cycle du carbone et dynamique des températures

Ce notebook met en œuvre un modèle climatique réduit inspiré de DICE. Il décrit le carbone atmosphérique et océanique, le forçage radiatif et les températures. Le pas de temps $\Delta$ vaut un an ou cinq ans, à partir de l’année `t0`.

## Variables d’état
- **$E(t)$** : émissions exogènes de CO₂, en GtC par an ;
- **$M_{AT}(t)$** : masse de carbone dans l’atmosphère, en GtC ;
- **$M_{UP}(t)$** : masse de carbone dans l’océan supérieur, en GtC ;
- **$M_{LO}(t)$** : masse de carbone dans l’océan profond, en GtC ;
- **$F(t)$** : forçage radiatif, en W/m² ;
- **$T_{AT}(t)$** : anomalie de température atmosphérique, en °C ;
- **$T_{LO}(t)$** : anomalie de température de l’océan profond, en °C.

## Équations du cycle du carbone

$$
M_{AT}(t) = (1 - \Delta b_{12}) M_{AT}(t-1)
+ \Delta b_{12} \frac{M_{AT0}}{M_{UP0}} M_{UP}(t-1)
+ \xi_E(t-1)
$$

$$
M_{UP}(t) = \Delta b_{12} M_{AT}(t-1)
+ \left(1 - \Delta b_{12} \tfrac{M_{AT0}}{M_{UP0}} - \Delta b_{23}\right) M_{UP}(t-1)
+ \Delta b_{23} \frac{M_{UP0}}{M_{LO0}} M_{LO}(t-1)
$$

$$
M_{LO}(t) = \Delta b_{23} M_{UP}(t-1)
+ \left(1 - \Delta b_{23} \tfrac{M_{UP0}}{M_{LO0}}\right) M_{LO}(t-1)
$$

## Forçage radiatif

$$
F(t) = F_{2 \times CO_2}\frac{\ln\!\left(M_{AT}(t-1)/M_{AT}^{\text{pré}}\right)}{\ln(2)}
$$

où $F_{2 \times CO_2}$ est le forçage produit par un doublement du CO₂ et $M_{AT}^{\text{pré}}$ la masse atmosphérique préindustrielle.

## Dynamique des températures

$$
T_{AT}(t) = T_{AT}(t-1)
+ \Delta c_1 F(t)
- \Delta c_1 \frac{F_{2 \times CO_2}}{T_{2 \times CO_2}} T_{AT}(t-1)
- \Delta c_1 c_3 \big(T_{AT}(t-1)-T_{LO}(t-1)\big),
$$

$$
T_{LO}(t) = T_{LO}(t-1)
+ \Delta c_4 \big(T_{AT}(t-1)-T_{LO}(t-1)\big).
$$

## Paramètres principaux
- $\Delta$ : pas de temps, en années ;
- $F_{2 \times CO_2}$ : forçage associé au doublement du CO₂ ;
- $T_{2 \times CO_2}$ : sensibilité climatique à l’équilibre ;
- $b_{12},b_{23}$ : taux de transfert du carbone entre atmosphère et océans ;
- $c_1,c_3,c_4$ : coefficients de transfert thermique ;
- $\xi$ : facteur de conversion du CO₂ en carbone.

## Objectifs
1. Initialiser les variables d’état à l’année `t0`.
2. Simuler le carbone et les températures au cours du temps.
3. Étudier l’effet de plusieurs scénarios d’émissions.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
if not Path("climate_models.py").exists() and Path("../climate_models.py").exists():
    sys.path.insert(0, str(Path("..").resolve()))
import climate_models as CM

# A — Dynamique climatique historique

Nous utilisons les émissions annuelles de CO₂ par pays publiées par *Our World in Data*, depuis le début de l’industrialisation.


In [ ]:
import pandas as pd
import requests

# Va chercher les données.
df = pd.read_csv("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Récupération des métadonnées
metadata = requests.get("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()



In [ ]:
# Select World data
df_world = df[df["entity"] == "World"]

# Ou en utilisant la colonne de code
df_world = df[df["code"] == "OWID_WRL"]

# Convert into Tons into GtCO2
world_co2 = df_world["emissions_total"]/(10**9)
year_co2  = df_world["year"]
print(df_world.head())




### A-1) Tracer les trajectoires des émissions de carbone


In [ ]:

plt.plot(year_co2,world_co2)
plt.ylabel('gigatons carbon')
plt.show()


### A-2) Simuler la dynamique climatique

#### Initialisation du système

Le modèle démarre à l’équilibre préindustriel, vers 1750, avant l’essor des émissions industrielles de CO₂.

- Les stocks de carbone sont fixés à leurs niveaux préindustriels :
  - $M_{AT0}=M_{AT}^{\text{pré}}$ dans l’atmosphère ;
  - $M_{UP0}=M_{UP}^{\text{pré}}$ dans l’océan supérieur ;
  - $M_{LO0}=M_{LO}^{\text{pré}}$ dans l’océan profond.
- Les anomalies de température sont initialement nulles : $T_{AT0}=T_{LO0}=0$.
- La simulation annuelle couvre 1749–2020.


In [ ]:
# Créer l’étalonnage principal par défaut
p       = CM.Params()
    
# Time definitions
p.Delta = 1         # Annual Data
p.t0    = 1749      #
p.tT    = 2024
p.nT    = (p.tT + p.Delta - p.t0)//p.Delta # update number of periods
    
# Initialisation au carbone préindustriel
p.M_AT0 = p.mat
p.M_UP0 = p.mup
p.M_LO0 = p.mlo

# No Temperatures anomalies
p.T_AT0 = 0
p.T_LO0 = 0

# No methan 
p.M_CH40 = p.mch4

# Initialiser la matrice Temps x Variables et initialiser les variables d'état
path = CM.init_states(p)

# Par défaut : pas d'émissions
# Mettre à jour le modèle (sans émissions)
path = CM.update_path(path,p,1750,2020)

# convert into data frame
df = CM.mat_to_df(path,p)


# Show outcome
df

**Exercice**
1. Tracez le carbone atmosphérique `M_AT` et la température `T_AT`.
2. Commentez brièvement les résultats.


In [ ]:
plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.plot(path[:,p.i_time],path[:,p.i_T_AT], label="Atmospheric Temperature")
plt.title("Global Temperatures")
plt.subplot(1,2,2)
plt.plot(path[:,p.i_time],path[:,p.i_M_AT], label="Atmospheric CO₂")
plt.legend()
plt.xlabel("Année")
plt.title("Atmospheric Carbon")
plt.show()

> Vous avez écrit la réponse ici.

En l'absence d'émissions de GES, il n'y a pas de changement climatique.

### A-3) Alimenter le modèle avec les émissions historiques

**Objectif.** Charger les émissions historiques de CO₂, les convertir en GtC/an, les injecter dans `path[:, p.i_E]`, simuler la trajectoire climatique, puis commenter les résultats.

**Exercice**
1. Affectez à `path[:, p.i_E]` la série `world_co2`, alignée sur les années, puis exécutez :
   ```python
   CM.update_path(path, p, 1750, 2020)
   ```
2. Tracez l’accumulation du carbone atmosphérique et l’anomalie de température mondiale.


In [ ]:
# Your code here.
year_co2       = year_co2.reset_index(drop=True)
world_co2      = world_co2.reset_index(drop=True)
tt             = np.where(year_co2 == 2024)[0][0]
path_pollution = path.copy()
pass


path_pollution[1:, p.i_E] = world_co2[0:(tt+1)]
path_pollution            = CM.update_path(path_pollution,p,1750,2024)

plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.plot(path_pollution[:,p.i_time],path_pollution[:,p.i_T_AT], label="Atmospheric Temperature")
plt.title("Global Temperatures")
plt.subplot(1,2,2)
plt.plot(path_pollution[:,p.i_time],path_pollution[:,p.i_M_AT], label="Atmospheric CO₂")
plt.legend()
plt.xlabel("Année")
plt.title("Atmospheric Carbon")
plt.show()

print(path_pollution[tt,p.i_T_AT])
print(path_pollution[tt,p.i_M_LO])

print(path_pollution[tt,p.i_T_AT])



### A-4) Comparer les anomalies de température observées et simulées

Comparez l’anomalie mondiale observée près de la surface avec l’anomalie atmosphérique simulée à partir des émissions historiques. Discutez les niveaux, les tendances et la variabilité.

1. **Comparaison graphique.** Tracez `world_temp` en fonction de `world_temp_yr` et `path_pollution[:, p.i_T_AT]`, en alignant les deux séries sur les mêmes années. Ajoutez une légende, un titre et une grille.
2. **Diagnostic.**
   - Le modèle reproduit-il la tendance observée ?
   - Les écarts diffèrent-ils entre le début et la fin de l’échantillon ?
   - La série observée contient une variabilité de court terme — ENSO, volcans — absente du modèle réduit.
   - Le modèle sous-estime-t-il ou surestime-t-il l’ampleur du réchauffement ?


In [ ]:
import pandas as pd
import requests

# Va chercher les données.
temperatures_data = pd.read_csv("https://ourworldindata.org/grapher/temperature-anomaly.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Select column World
temperatures_world = temperatures_data[temperatures_data["code"] == "OWID_WRL"]

# Certaines années et températures mondiales
world_temp    = temperatures_world["near_surface_temperature_anomaly"]
world_temp    = world_temp.reset_index(drop=True)
world_temp_yr = temperatures_world["year"]
world_temp_yr = world_temp_yr.reset_index(drop=True)

pass
plt.figure(figsize=(5,4))
plt.subplot(1,1,1)
plt.plot(path_pollution[:,p.i_time], path_pollution[:,p.i_T_AT], label="Simulated")
plt.plot(world_temp_yr, world_temp, label="Observed")
plt.title("Global Temperatures")
plt.legend()
plt.show()

> Vous avez écrit la réponse ici.

### A-5) Ajouter le cycle du méthane

Le méthane (CH₄) a une durée de vie plus courte que le CO₂, mais un pouvoir réchauffant plus élevé par molécule. Une équation simple décrit son accumulation et sa décomposition :

$$
M_{CH4}(t) = (1-\delta\Delta)\big(M_{CH4}(t-1)-m_{CH4}\big)+\Delta E_{CH4}(t).
$$

- $M_{CH4}(t)$ : concentration atmosphérique de méthane, en ppb ;
- $E_{CH4}(t)$ : émissions de méthane, converties en ppb ;
- $\delta\simeq1/12$ : taux annuel de décomposition ;
- $m_{CH4}$ : niveau préindustriel.

Le forçage radiatif du méthane est approximé par

$$
F_{CH4}(t)=\alpha_{CH4}\left(\sqrt{M_{CH4}(t)}-\sqrt{M_{CH4,0}}\right),
$$

avec $M_{CH4,0}\simeq722$ ppb et $\alpha_{CH4}\simeq0{,}036\,\text{W/m}^2/\sqrt{\text{ppb}}$.

Le forçage total devient

$$
F(t)=F_{CO2}(t)+F_{CH4}(t)+F_{\text{ex}}(t),
$$

où $F_{\text{ex}}$, fixé ici à zéro, représente les autres forçages.

**Travail demandé**
- Téléchargez les données de méthane et alimentez le modèle.
- Adaptez `update_path` pour simuler le stock de méthane et son forçage.
- Comparez, dans une figure à trois panneaux, la température mondiale, le méthane atmosphérique et le carbone atmosphérique.
- Commentez vos résultats.


In [ ]:
#Your code here
import pandas as pd
import requests

# Va chercher les données.
df_ch4 = pd.read_csv("https://ourworldindata.org/grapher/ghg-emissions-by-gas.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
df_ch4_world = df_ch4[df_ch4["entity"] == "World"]

print(df_ch4_world)

# Convert ton equivalent tEqCO2 = tCH4*29.8
#                        Mt     = 10^6 tons
#                        ppb    = 2.78*Mt
world_ch4 = df_ch4_world["annual_emissions_ch4_total_co2eq"]/(10**6 * 29.8 * 2.78)
year_ch4  = df_ch4_world["year"]

# Drop indices
year_ch4       = year_ch4.reset_index(drop=True)
world_ch4      = world_ch4.reset_index(drop=True)
tTmh4          = np.where(year_ch4 == 2020)[0][0]
path_ch4_co2   = path_pollution.copy()

pass



t0             = np.where(path_ch4_co2[:,p.i_time] == 1850)[0][0]
t_ch4_end = np.where(path_ch4_co2[:, p.i_time] == 2020)[0][0]
path_ch4_co2[t0:t_ch4_end + 1, p.i_E_CH4] = world_ch4[0:(tTmh4+1)]
path_ch4_co2            = CM.update_path(path_ch4_co2,p,1750,2020)

print(path_ch4_co2[t0:t_ch4_end + 1, p.i_E_CH4])
plt.figure(figsize=(10,6))
plt.subplot(1,3,1)
plt.plot(path_pollution[:,p.i_time],path_pollution[:,p.i_T_AT], label="Carbon only")
plt.plot(path_ch4_co2[:,p.i_time],path_ch4_co2[:,p.i_T_AT], label="CO₂ + MH_{4}")
plt.plot(world_temp_yr,world_temp, label="Observed")
plt.title("Global Temperatures")
plt.legend()
plt.subplot(1,3,2)
plt.plot(path_pollution[:,p.i_time],path_pollution[:,p.i_M_AT])
plt.plot(path_ch4_co2[:,p.i_time],path_ch4_co2[:,p.i_M_AT])
plt.title("Atmospheric carbon (GtC)")
plt.subplot(1,3,3)
plt.plot(path_pollution[:,p.i_time],path_pollution[:,p.i_M_CH4])
plt.plot(path_ch4_co2[:,p.i_time],path_ch4_co2[:,p.i_M_CH4])
plt.xlabel("Année")
plt.title("Atmospheric methan (ppb)")
plt.show()


> Vous avez écrit la réponse ici.

# B — Projeter les températures avec les scénarios SSP

Nous utilisons les trajectoires socio-économiques partagées (SSP) du GIEC comme scénarios conditionnels d’émissions. Les données décennales sont interpolées sur une grille annuelle pour être compatibles avec le modèle.

**Source des données**

[Our World in Data — explorateur des scénarios du GIEC](https://ourworldindata.org/explorers/ipcc-scenarios?Metric=Greenhouse+gas+emissions&Sub-metric=Carbon+dioxide+%28CO%E2%82%82%29&Region=Global&country=SSP1+-+Baseline~SSP2+-+Baseline~SSP3+-+Baseline~SSP4+-+Baseline~SSP5+-+Baseline)

Le jeu de données contient cinq trajectoires de référence :
- **SSP1 — Durabilité** ;
- **SSP2 — Voie médiane** ;
- **SSP3 — Rivalités régionales** ;
- **SSP4 — Inégalités** ;
- **SSP5 — Développement fondé sur les énergies fossiles**.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger
data_file = Path("Notebook_ClimateModels_SSP_data.csv")
if not data_file.exists():
    data_file = Path("../Notebook_ClimateModels_SSP_data.csv")
df = pd.read_csv(data_file, sep=";", decimal=",")

# Choisir année + colonnes SSP
year = df["Year"]

# Emplacement
plt.figure

plt.plot(year, df["SSP1"], label="SSP1",color="#1B9E77")
plt.plot(year, df["SSP2"], label="SSP2",color="#D95F02")
plt.plot(year, df["SSP3"], label="SSP3",color="#8C6D31")
plt.plot(year, df["SSP4"], label="SSP4",color="#2CA25F")
plt.plot(year, df["SSP5"], label="SSP5",color="#6F42C1")
plt.grid(True, alpha=0.3)
plt.legend(title="Scenario", ncol=2, frameon=False)
plt.title("Per capita CO₂ emissions — SSP baselines (global)")
plt.xlabel("Année"); plt.ylabel("CO₂ per capita")
plt.tight_layout();
plt.show()

### B-1) Simuler les températures selon les scénarios SSP

Utilisez les émissions annuelles interpolées de 2020 à 2100 comme entrées du modèle, puis comparez les anomalies de température atmosphérique `T_AT`.

1. Initialisez le modèle en 2020 avec le dernier état de la simulation historique.
2. Pour chaque SSP, renseignez la colonne des émissions `E` entre 2020 et 2100.
3. Exécutez `CM.update_path`.
4. Tracez ensemble les températures obtenues pour les cinq SSP.

**Question.** Comparez et commentez les trajectoires simulées.


In [ ]:
# Your code here
# Créer l’étalonnage principal par défaut
p2       = CM.Params()
    
# Time definitions
p2.Delta = 1         # Annual Data
p2.t0    = 1749      #
p2.tT    = 2100
p2.nT    = (p2.tT + p2.Delta - p2.t0)//p.Delta # update number of periods
    
# Initialisation au carbone préindustriel
p2.M_AT0 = p.mat
p2.M_UP0 = p.mup
p2.M_LO0 = p.mlo
# No Temperatures anomalies
p2.T_AT0 = 0
p2.T_LO0 = 0

# Initialiser la matrice Temps x Variables et initialiser les variables d'état
path_SSP = CM.init_states(p2)

print(path_SSP)
# Par défaut : pas d'émissions
# Mettre à jour le modèle (sans émissions)
path_SSP = CM.update_path(path_SSP,p2,1750,2020)

path_SSP1 = path_SSP.copy()
path_SSP2 = path_SSP.copy()
path_SSP3 = path_SSP.copy()
path_SSP4 = path_SSP.copy()
path_SSP5 = path_SSP.copy()

pass


tt             = np.where(year_co2 == 2020)[0][0]
path_SSP1[(1+tt):, p2.i_E] = df["SSP1"]
path_SSP1 = CM.update_path(path_SSP1,p2,2020,2100)

path_SSP2[(1+tt):, p2.i_E] = df["SSP2"]
path_SSP2 = CM.update_path(path_SSP2,p2,2020,2100)

path_SSP3[(1+tt):, p2.i_E] = df["SSP3"]
path_SSP3 = CM.update_path(path_SSP3,p2,2020,2100)

path_SSP4[(1+tt):, p2.i_E] = df["SSP4"]
path_SSP4 = CM.update_path(path_SSP4,p2,2020,2100)

path_SSP5[(1+tt):, p2.i_E] = df["SSP5"]
path_SSP5 = CM.update_path(path_SSP5,p2,2020,2100)

plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.plot(path_SSP1[:,p2.i_time],path_SSP1[:,p2.i_T_AT], label="SSP1",color="#1B9E77")
plt.plot(path_SSP2[:,p2.i_time],path_SSP2[:,p2.i_T_AT], label="SSP2",color="#D95F02")
plt.plot(path_SSP3[:,p2.i_time],path_SSP3[:,p2.i_T_AT], label="SSP3",color="#8C6D31")
plt.plot(path_SSP4[:,p2.i_time],path_SSP4[:,p2.i_T_AT], label="SSP4",color="#2CA25F")
plt.plot(path_SSP5[:,p2.i_time],path_SSP5[:,p2.i_T_AT], label="SSP5",color="#6F42C1")
plt.xlim(2020, 2100) 
plt.title("Global Temperatures")
plt.subplot(1,2,2)
plt.plot(path_SSP1[:,p2.i_time],path_SSP1[:,p2.i_M_AT], label="SSP1",color="#1B9E77")
plt.plot(path_SSP2[:,p2.i_time],path_SSP2[:,p2.i_M_AT], label="SSP2",color="#D95F02")
plt.plot(path_SSP3[:,p2.i_time],path_SSP3[:,p2.i_M_AT], label="SSP3",color="#8C6D31")
plt.plot(path_SSP4[:,p2.i_time],path_SSP4[:,p2.i_M_AT], label="SSP4",color="#2CA25F")
plt.plot(path_SSP5[:,p2.i_time],path_SSP5[:,p2.i_M_AT], label="SSP5",color="#6F42C1")
plt.xlim(2020, 2100) 
plt.legend()
plt.xlabel("Année")
plt.title("Atmospheric Carbon")
plt.show()

> Vous avez écrit la réponse ici.